In [ ]:
# ============================================================
# MODELO FINAL — RANDOM FOREST + RISCO HISTÓRICO MUNICIPAL
# RAIS_BASE_MODELO_FINAL_V3
# ============================================================
#
# OBJETIVOS
# ------------------------------------------------------------
# 1. Reproduzir o Random Forest original como baseline.
# 2. Treinar a especificação final:
#       RF original
#       + Risco_historico_municipio
#       + Log_n_historico_municipio
# 3. Construir o risco municipal SEM informação futura.
# 4. Validar ambos os modelos em 2024.
# 5. Definir o limiar de maior F1 exclusivamente em 2024.
# 6. Congelar modelo, preprocessamento e limiar.
# 7. SOMENTE DEPOIS abrir 2025.
# 8. Avaliar temporalmente baseline e modelo final em 2025.
# 9. Refazer calibração.
# 10. Refazer SHAP do modelo final.
# 11. Gerar ranking municipal completo + Top 30 / Bottom 30.
# 12. Gerar arquivos para Resultados e Discussão do TCC.
#
# IMPORTANTE
# ------------------------------------------------------------
# Risco municipal temporal:
#
# treino:
#   2020 -> sem histórico anterior
#   2021 -> 2020
#   2022 -> 2020-2021
#   2023 -> 2020-2022
#
# validação:
#   2024 -> 2020-2023
#
# avaliação posterior:
#   2025 -> 2020-2024
#
# Fórmula:
#
#   R_m = (S_m + alpha * taxa_UF) / (N_m + alpha)
#
# alpha = 1000
#
# O município NÃO entra diretamente como código ou dummy.
# ============================================================


# ============================================================
# 1. IMPORTAÇÕES
# ============================================================

from google.colab import drive

drive.mount(
    "/content/drive",
    force_remount=False
)

import gc
import glob
import json
import os
import time
import warnings

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import requests

from IPython.display import display

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    brier_score_loss,
    confusion_matrix,
    f1_score,
    log_loss,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

warnings.filterwarnings(
    "ignore",
    category=FutureWarning
)


# ============================================================
# 2. CONFIGURAÇÕES
# ============================================================

SEED = 42

TARGET = "Y_doenca"

ANOS_TREINO = [
    2020,
    2021,
    2022,
    2023,
]

ANO_VALIDACAO = 2024

ANO_AVALIACAO = 2025


PASTA_TCC = (
    "/content/drive/MyDrive/TCC_2"
)

PASTA_BASE = os.path.join(
    PASTA_TCC,
    "dados",
    "RAIS_BASE_MODELO_FINAL_V3"
)

PASTA_RESULTADOS = os.path.join(
    PASTA_TCC,
    "resultados",
    "MODELO_FINAL_RF_RISCO_MUNICIPAL"
)

PASTA_MODELOS = os.path.join(
    PASTA_RESULTADOS,
    "modelos"
)

PASTA_TABELAS = os.path.join(
    PASTA_RESULTADOS,
    "tabelas"
)

PASTA_FIGURAS = os.path.join(
    PASTA_RESULTADOS,
    "figuras"
)

PASTA_PREDICOES = os.path.join(
    PASTA_RESULTADOS,
    "predicoes"
)

for pasta in [
    PASTA_RESULTADOS,
    PASTA_MODELOS,
    PASTA_TABELAS,
    PASTA_FIGURAS,
    PASTA_PREDICOES,
]:

    os.makedirs(
        pasta,
        exist_ok=True
    )


# ------------------------------------------------------------
# Processamento
# ------------------------------------------------------------

BATCH_SIZE = 500_000

BATCH_VALIDACAO = 200_000

SHAP_N = 10_000

TOP_N_MUNICIPIOS = 30

MIN_N_RANKING = 500

ALPHA_MUNICIPIO = 1000.0


# ============================================================
# 3. VARIÁVEIS
# ============================================================

COL_MUN = (
    "Municipio_estabelecimento_codigo"
)

COL_RISCO = (
    "Risco_historico_municipio"
)

COL_LOGN = (
    "Log_n_historico_municipio"
)


NUMERICAS_BASE = [
    "Idade",
    "Qtd_horas_contratuais",
    "Tempo_emprego_meses",
]


CATEGORICAS_BASE = [
    "UF",
    "Familia_CBO",
    "Sexo_codigo",
    "Tipo_vinculo_macro",
    "Natureza_macro",
    "Tamanho_estabelecimento_codigo",
    "Indicador_deficiencia_codigo",
]


NUMERICAS_FINAL = (
    NUMERICAS_BASE
    +
    [
        COL_RISCO,
        COL_LOGN,
    ]
)


CATEGORICAS_FINAL = (
    CATEGORICAS_BASE.copy()
)


COLUNAS_LER = sorted(
    set(
        NUMERICAS_BASE
        +
        CATEGORICAS_BASE
        +
        [
            COL_MUN,
            TARGET,
        ]
    )
)


# ============================================================
# 4. RANDOM FOREST — MESMOS HIPERPARÂMETROS
# ============================================================

RF_PARAMS = dict(

    n_estimators=200,

    max_depth=24,

    min_samples_split=40,

    min_samples_leaf=20,

    max_features="sqrt",

    max_samples=0.25,

    class_weight="balanced_subsample",

    n_jobs=-1,

    random_state=SEED,

    verbose=1,
)


# ============================================================
# 5. FUNÇÕES DE LIMPEZA
# ============================================================

def limpar_categoria(
    serie
):

    return (
        serie
        .astype("string")
        .str.strip()
        .str.replace(
            r"\s+",
            " ",
            regex=True
        )
        .fillna(
            "__MISSING__"
        )
        .replace(
            {
                "":
                    "__MISSING__",

                "<NA>":
                    "__MISSING__",

                "nan":
                    "__MISSING__",

                "None":
                    "__MISSING__",
            }
        )
    )


def normalizar_municipio(
    serie
):

    return (
        serie
        .astype("string")
        .str.strip()
        .str.extract(
            r"(\d{6})",
            expand=False
        )
        .fillna(
            "__MISSING__"
        )
    )


# ============================================================
# 6. LOCALIZAR ARQUIVOS
# ============================================================

def localizar_arquivos_ano(
    ano
):

    arquivos = sorted(
        glob.glob(
            os.path.join(
                PASTA_BASE,
                "**",
                f"RAIS_MODELO_FINAL_V3_{ano}_*.parquet"
            ),
            recursive=True
        )
    )


    if len(
        arquivos
    ) != 6:

        raise RuntimeError(
            f"{ano}: esperava 6 arquivos, "
            f"mas encontrei {len(arquivos)}."
        )


    return arquivos


# ------------------------------------------------------------
# IMPORTANTE:
# Inicialmente localizamos SOMENTE 2020–2024.
# 2025 será localizado apenas depois do congelamento.
# ------------------------------------------------------------

ARQUIVOS = {}


for ano in [
    2020,
    2021,
    2022,
    2023,
    2024,
]:

    ARQUIVOS[
        ano
    ] = localizar_arquivos_ano(
        ano
    )


print(
    "Arquivos 2020–2024 localizados."
)

print(
    "2025 ainda NÃO foi acessado."
)


# ============================================================
# 7. VALIDAR SCHEMA
# ============================================================

for ano, arquivos in (
    ARQUIVOS.items()
):

    for arquivo in arquivos:

        schema = set(
            pq.ParquetFile(
                arquivo
            )
            .schema_arrow
            .names
        )


        faltantes = (
            set(
                COLUNAS_LER
            )
            -
            schema
        )


        if faltantes:

            raise RuntimeError(
                f"{ano}: faltam colunas: "
                f"{sorted(faltantes)}"
            )


print(
    "Schema V3: OK."
)


# ============================================================
# 8. ESTATÍSTICAS MUNICIPAIS DE UM ANO
# ============================================================

def calcular_estatisticas_municipais_ano(
    ano
):

    partes = []


    for arquivo in ARQUIVOS[
        ano
    ]:

        parquet = pq.ParquetFile(
            arquivo
        )


        for batch in parquet.iter_batches(

            batch_size=BATCH_SIZE,

            columns=[
                "UF",
                COL_MUN,
                TARGET,
            ]
        ):

            df = batch.to_pandas()


            df[
                "UF"
            ] = limpar_categoria(
                df[
                    "UF"
                ]
            )


            df[
                COL_MUN
            ] = normalizar_municipio(
                df[
                    COL_MUN
                ]
            )


            df[
                TARGET
            ] = pd.to_numeric(
                df[
                    TARGET
                ],
                errors="raise"
            )


            resumo = (
                df
                .groupby(
                    [
                        "UF",
                        COL_MUN,
                    ],
                    as_index=False
                )[
                    TARGET
                ]
                .agg(
                    N="size",
                    S="sum"
                )
            )


            partes.append(
                resumo
            )


            del df

            gc.collect()


    resultado = (
        pd.concat(
            partes,
            ignore_index=True
        )
        .groupby(
            [
                "UF",
                COL_MUN,
            ],
            as_index=False
        )
        .agg(
            N=("N", "sum"),
            S=("S", "sum"),
        )
    )


    resultado[
        "Ano"
    ] = ano


    return resultado


# ============================================================
# 9. ESTATÍSTICAS 2020–2023
# ============================================================

ESTATISTICAS_ANUAIS = {}


for ano in ANOS_TREINO:

    print(
        f"Estatísticas municipais: {ano}"
    )

    ESTATISTICAS_ANUAIS[
        ano
    ] = calcular_estatisticas_municipais_ano(
        ano
    )


# ============================================================
# 10. COMBINAR HISTÓRICO
# ============================================================

def combinar_historico(
    anos
):

    if not anos:

        return None


    df = pd.concat(
        [
            ESTATISTICAS_ANUAIS[
                ano
            ][
                [
                    "UF",
                    COL_MUN,
                    "N",
                    "S",
                ]
            ]

            for ano in anos
        ],
        ignore_index=True
    )


    municipios = (
        df
        .groupby(
            [
                "UF",
                COL_MUN,
            ],
            as_index=False
        )
        .agg(
            N=("N", "sum"),
            S=("S", "sum"),
        )
    )


    uf = (
        municipios
        .groupby(
            "UF",
            as_index=False
        )
        .agg(
            N_UF=("N", "sum"),
            S_UF=("S", "sum"),
        )
    )


    uf[
        "Taxa_UF"
    ] = (
        uf[
            "S_UF"
        ]
        /
        uf[
            "N_UF"
        ]
    )


    taxa_nacional = (
        municipios[
            "S"
        ].sum()
        /
        municipios[
            "N"
        ].sum()
    )


    municipios = municipios.merge(
        uf[
            [
                "UF",
                "Taxa_UF",
            ]
        ],
        on="UF",
        how="left"
    )


    municipios[
        "Taxa_bruta"
    ] = (
        municipios[
            "S"
        ]
        /
        municipios[
            "N"
        ]
    )


    municipios[
        COL_RISCO
    ] = (
        municipios[
            "S"
        ]
        +
        ALPHA_MUNICIPIO
        *
        municipios[
            "Taxa_UF"
        ]
    ) / (
        municipios[
            "N"
        ]
        +
        ALPHA_MUNICIPIO
    )


    municipios[
        COL_LOGN
    ] = np.log1p(
        municipios[
            "N"
        ]
    )


    return (
        municipios,
        uf,
        float(
            taxa_nacional
        ),
    )


# ============================================================
# 11. APLICAR RISCO HISTÓRICO
# ============================================================

def adicionar_risco_municipal(
    df,
    anos_historico
):

    # Primeiro ano: não existe histórico anterior
    if not anos_historico:

        df[
            COL_RISCO
        ] = np.nan

        df[
            COL_LOGN
        ] = np.float32(
            0
        )

        return df


    (
        municipios,
        uf,
        taxa_nacional,
    ) = combinar_historico(
        anos_historico
    )


    chave_municipio = (
        municipios
        .set_index(
            [
                "UF",
                COL_MUN,
            ]
        )
    )


    taxa_uf = (
        uf
        .set_index(
            "UF"
        )[
            "Taxa_UF"
        ]
    )


    uf_linha = (
        limpar_categoria(
            df[
                "UF"
            ]
        )
    )


    mun_linha = (
        normalizar_municipio(
            df[
                COL_MUN
            ]
        )
    )


    idx = pd.MultiIndex.from_arrays(
        [
            uf_linha,
            mun_linha,
        ],
        names=[
            "UF",
            COL_MUN,
        ]
    )


    risco = (
        chave_municipio[
            COL_RISCO
        ]
        .reindex(
            idx
        )
        .to_numpy()
    )


    logn = (
        chave_municipio[
            COL_LOGN
        ]
        .reindex(
            idx
        )
        .to_numpy()
    )


    # --------------------------------------------------------
    # Município novo:
    # fallback para taxa da UF.
    # --------------------------------------------------------

    taxa_uf_linha = (
        uf_linha
        .map(
            taxa_uf
        )
        .fillna(
            taxa_nacional
        )
        .to_numpy(
            dtype=float
        )
    )


    risco = np.where(
        pd.isna(
            risco
        ),
        taxa_uf_linha,
        risco
    )


    logn = np.where(
        pd.isna(
            logn
        ),
        0,
        logn
    )


    df[
        COL_RISCO
    ] = risco.astype(
        "float32"
    )


    df[
        COL_LOGN
    ] = logn.astype(
        "float32"
    )


    return df


# ============================================================
# 12. DESCOBRIR CATEGORIAS DO TREINO
# ============================================================

def descobrir_categorias_treino():

    conjuntos = {
        coluna:
            {
                "__MISSING__"
            }

        for coluna
        in CATEGORICAS_BASE
    }


    for ano in ANOS_TREINO:

        for arquivo in ARQUIVOS[
            ano
        ]:

            parquet = pq.ParquetFile(
                arquivo
            )


            for batch in parquet.iter_batches(

                batch_size=BATCH_SIZE,

                columns=CATEGORICAS_BASE
            ):

                df = batch.to_pandas()


                for coluna in (
                    CATEGORICAS_BASE
                ):

                    valores = limpar_categoria(
                        df[
                            coluna
                        ]
                    )


                    conjuntos[
                        coluna
                    ].update(
                        valores
                        .astype(str)
                        .unique()
                        .tolist()
                    )


                del df

                gc.collect()


    return {
        coluna:
            sorted(
                valores
            )

        for coluna, valores
        in conjuntos.items()
    }


print(
    "Descobrindo categorias..."
)


CATEGORIAS_TREINO = (
    descobrir_categorias_treino()
)


# ============================================================
# 13. CARREGAR TREINO 2020–2023
# ============================================================

def preparar_chunk(
    df,
    ano,
    anos_historico
):

    df[
        "Ano"
    ] = np.int16(
        ano
    )


    df[
        TARGET
    ] = (
        pd.to_numeric(
            df[
                TARGET
            ],
            errors="raise"
        )
        .astype(
            "int8"
        )
    )


    for coluna in NUMERICAS_BASE:

        df[
            coluna
        ] = (
            pd.to_numeric(
                df[
                    coluna
                ],
                errors="coerce"
            )
            .astype(
                "float32"
            )
        )


    df[
        "UF"
    ] = limpar_categoria(
        df[
            "UF"
        ]
    )


    df[
        COL_MUN
    ] = normalizar_municipio(
        df[
            COL_MUN
        ]
    )


    df = adicionar_risco_municipal(
        df,
        anos_historico
    )


    for coluna in CATEGORICAS_BASE:

        valores = limpar_categoria(
            df[
                coluna
            ]
        )


        df[
            coluna
        ] = pd.Categorical(
            valores,
            categories=CATEGORIAS_TREINO[
                coluna
            ]
        )


    return df


partes_treino = []

total_treino = 0


for ano in ANOS_TREINO:

    anos_historico = [
        a
        for a in ANOS_TREINO
        if a < ano
    ]


    print(
        f"\nTreino {ano} | histórico: "
        f"{anos_historico}"
    )


    for arquivo in ARQUIVOS[
        ano
    ]:

        parquet = pq.ParquetFile(
            arquivo
        )


        for batch in parquet.iter_batches(

            batch_size=BATCH_SIZE,

            columns=COLUNAS_LER
        ):

            df = batch.to_pandas()


            df = preparar_chunk(
                df,
                ano,
                anos_historico
            )


            partes_treino.append(
                df
            )


            total_treino += len(
                df
            )


            print(
                f"{total_treino:,}",
                end="\r"
            )


df_treino = pd.concat(
    partes_treino,
    ignore_index=True
)


del partes_treino

gc.collect()


print(
    "\nTreino completo:",
    f"{len(df_treino):,}"
)


print(
    "Prevalência:",
    f"{df_treino[TARGET].mean():.4%}"
)


# ============================================================
# 14. AUDITORIA TEMPORAL DO RISCO
# ============================================================

auditoria_risco = []


for ano in ANOS_TREINO:

    sub = df_treino[
        df_treino[
            "Ano"
        ]
        ==
        ano
    ]


    auditoria_risco.append(
        {
            "Ano":
                ano,

            "Historico_utilizado":
                ",".join(
                    map(
                        str,
                        [
                            x
                            for x in ANOS_TREINO
                            if x < ano
                        ]
                    )
                )
                or
                "nenhum",

            "N":
                int(
                    len(
                        sub
                    )
                ),

            "Prevalencia":
                float(
                    sub[
                        TARGET
                    ]
                    .mean()
                ),

            "Risco_municipal_medio":
                float(
                    sub[
                        COL_RISCO
                    ]
                    .mean()
                )
                if sub[
                    COL_RISCO
                ].notna().any()
                else np.nan,
        }
    )


pd.DataFrame(
    auditoria_risco
).to_csv(
    os.path.join(
        PASTA_TABELAS,
        "01_auditoria_risco_treino.csv"
    ),
    index=False,
    encoding="utf-8-sig"
)


# ============================================================
# 15. PREPROCESSADORES
# ============================================================

def criar_one_hot():

    try:

        return OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=True,
            dtype=np.float32
        )

    except TypeError:

        return OneHotEncoder(
            handle_unknown="ignore",
            sparse=True,
            dtype=np.float32
        )


def criar_preprocessador(
    numericas,
    categoricas
):

    return ColumnTransformer(
        [
            (
                "num",
                Pipeline(
                    [
                        (
                            "imputacao",
                            SimpleImputer(
                                strategy="median"
                            )
                        )
                    ]
                ),
                numericas
            ),

            (
                "cat",
                Pipeline(
                    [
                        (
                            "onehot",
                            criar_one_hot()
                        )
                    ]
                ),
                categoricas
            ),
        ],
        remainder="drop",
        sparse_threshold=1.0
    )


prep_baseline = criar_preprocessador(
    NUMERICAS_BASE,
    CATEGORICAS_BASE
)


prep_final = criar_preprocessador(
    NUMERICAS_FINAL,
    CATEGORICAS_FINAL
)


# ============================================================
# 16. TREINAR BASELINE
# ============================================================

print(
    "\nCriando matriz baseline..."
)


X_baseline = (
    prep_baseline
    .fit_transform(
        df_treino
    )
)


print(
    X_baseline.shape
)


rf_baseline = RandomForestClassifier(
    **RF_PARAMS
)


print(
    "Treinando baseline..."
)


rf_baseline.fit(
    X_baseline,
    df_treino[
        TARGET
    ].to_numpy(
        dtype=np.int8
    )
)


del X_baseline

gc.collect()


# ============================================================
# 17. TREINAR MODELO FINAL
# ============================================================

print(
    "\nCriando matriz modelo final..."
)


X_final = (
    prep_final
    .fit_transform(
        df_treino
    )
)


print(
    X_final.shape
)


rf_final = RandomForestClassifier(
    **RF_PARAMS
)


print(
    "Treinando modelo final..."
)


rf_final.fit(
    X_final,
    df_treino[
        TARGET
    ].to_numpy(
        dtype=np.int8
    )
)


del X_final

gc.collect()


# ============================================================
# 18. SALVAR MODELOS
# ============================================================

joblib.dump(
    {
        "modelo":
            rf_baseline,

        "preprocessador":
            prep_baseline,

        "numericas":
            NUMERICAS_BASE,

        "categoricas":
            CATEGORICAS_BASE,

        "anos_treino":
            ANOS_TREINO,

        "random_state":
            SEED,
    },
    os.path.join(
        PASTA_MODELOS,
        "RF_baseline.joblib"
    ),
    compress=3
)


joblib.dump(
    {
        "modelo":
            rf_final,

        "preprocessador":
            prep_final,

        "numericas":
            NUMERICAS_FINAL,

        "categoricas":
            CATEGORICAS_FINAL,

        "alpha_municipio":
            ALPHA_MUNICIPIO,

        "anos_treino":
            ANOS_TREINO,

        "risco_temporal":
            True,

        "random_state":
            SEED,
    },
    os.path.join(
        PASTA_MODELOS,
        "RF_final_risco_municipal.joblib"
    ),
    compress=3
)


# ============================================================
# 19. FUNÇÕES DE MÉTRICAS
# ============================================================

def melhor_threshold_f1(
    y,
    prob
):

    precision, recall, thresholds = (
        precision_recall_curve(
            y,
            prob
        )
    )


    denominador = (
        precision[:-1]
        +
        recall[:-1]
    )


    f1 = np.divide(
        2
        *
        precision[:-1]
        *
        recall[:-1],
        denominador,
        out=np.zeros_like(
            denominador
        ),
        where=(
            denominador
            >
            0
        )
    )


    indice = int(
        np.nanargmax(
            f1
        )
    )


    return float(
        thresholds[
            indice
        ]
    )


def threshold_para_recall(
    y,
    prob,
    recall_alvo
):

    precision, recall, thresholds = (
        precision_recall_curve(
            y,
            prob
        )
    )


    elegiveis = np.flatnonzero(
        recall[:-1]
        >=
        recall_alvo
    )


    if not len(
        elegiveis
    ):

        return 0.0


    indice = elegiveis[
        np.argmax(
            thresholds[
                elegiveis
            ]
        )
    ]


    return float(
        thresholds[
            indice
        ]
    )


def calcular_metricas(
    y,
    prob,
    threshold,
    modelo,
    ano
):

    pred = (
        prob
        >=
        threshold
    ).astype(
        np.int8
    )


    tn, fp, fn, tp = (
        confusion_matrix(
            y,
            pred,
            labels=[
                0,
                1,
            ]
        )
        .ravel()
    )


    especificidade = (
        tn
        /
        (
            tn
            +
            fp
        )
    )


    prevalencia = float(
        np.mean(
            y
        )
    )


    ap = float(
        average_precision_score(
            y,
            prob
        )
    )


    return {
        "Ano":
            ano,

        "Modelo":
            modelo,

        "Threshold":
            float(
                threshold
            ),

        "N":
            int(
                len(
                    y
                )
            ),

        "Prevalencia":
            prevalencia,

        "Average_Precision":
            ap,

        "AP_dividida_prevalencia":
            (
                ap
                /
                prevalencia
            ),

        "ROC_AUC":
            float(
                roc_auc_score(
                    y,
                    prob
                )
            ),

        "Accuracy":
            float(
                accuracy_score(
                    y,
                    pred
                )
            ),

        "Balanced_accuracy":
            float(
                balanced_accuracy_score(
                    y,
                    pred
                )
            ),

        "Precision":
            float(
                precision_score(
                    y,
                    pred,
                    zero_division=0
                )
            ),

        "Recall":
            float(
                recall_score(
                    y,
                    pred,
                    zero_division=0
                )
            ),

        "Especificidade":
            float(
                especificidade
            ),

        "F1":
            float(
                f1_score(
                    y,
                    pred,
                    zero_division=0
                )
            ),

        "TN":
            int(
                tn
            ),

        "FP":
            int(
                fp
            ),

        "FN":
            int(
                fn
            ),

        "TP":
            int(
                tp
            ),
    }


# ============================================================
# 20. PREDIZER UM ANO EM BATCHES
# ============================================================

def prever_ano(
    ano,
    arquivos,
    anos_historico,
    coletar_amostra=False
):

    y_partes = []

    prob_baseline_partes = []

    prob_final_partes = []

    amostra_partes = []


    # --------------------------------------------------------
    # Amostra SHAP com índices globais fixos
    # --------------------------------------------------------

    total_ano = sum(
        pq.ParquetFile(
            arquivo
        )
        .metadata
        .num_rows

        for arquivo in arquivos
    )


    if coletar_amostra:

        rng = np.random.default_rng(
            SEED
        )


        indices_amostra = np.sort(
            rng.choice(
                total_ano,
                size=min(
                    SHAP_N,
                    total_ano
                ),
                replace=False
            )
        )

    else:

        indices_amostra = np.array(
            [],
            dtype=int
        )


    posicao_global = 0


    for arquivo in arquivos:

        parquet = pq.ParquetFile(
            arquivo
        )


        for batch in parquet.iter_batches(

            batch_size=BATCH_VALIDACAO,

            columns=COLUNAS_LER
        ):

            df = batch.to_pandas()


            n_batch = len(
                df
            )


            df = preparar_chunk(
                df,
                ano,
                anos_historico
            )


            y = (
                df[
                    TARGET
                ]
                .to_numpy(
                    dtype=np.int8
                )
            )


            Xb = prep_baseline.transform(
                df
            )


            Xf = prep_final.transform(
                df
            )


            pb = (
                rf_baseline
                .predict_proba(
                    Xb
                )[
                    :,
                    1
                ]
                .astype(
                    np.float32
                )
            )


            pf = (
                rf_final
                .predict_proba(
                    Xf
                )[
                    :,
                    1
                ]
                .astype(
                    np.float32
                )
            )


            y_partes.append(
                y
            )


            prob_baseline_partes.append(
                pb
            )


            prob_final_partes.append(
                pf
            )


            # ------------------------------------------------
            # Coletar amostra para SHAP
            # ------------------------------------------------

            if coletar_amostra:

                ini = posicao_global

                fim = (
                    posicao_global
                    +
                    n_batch
                )


                esquerda = np.searchsorted(
                    indices_amostra,
                    ini,
                    side="left"
                )


                direita = np.searchsorted(
                    indices_amostra,
                    fim,
                    side="left"
                )


                selecionados = (
                    indices_amostra[
                        esquerda:
                        direita
                    ]
                    -
                    ini
                )


                if len(
                    selecionados
                ):

                    amostra_partes.append(
                        df.iloc[
                            selecionados
                        ].copy()
                    )


            posicao_global += (
                n_batch
            )


            print(
                f"{ano}: "
                f"{posicao_global:,}/"
                f"{total_ano:,}",
                end="\r"
            )


            del (
                df,
                Xb,
                Xf,
                y,
                pb,
                pf,
            )


            gc.collect()


    print()


    y_total = np.concatenate(
        y_partes
    )


    pb_total = np.concatenate(
        prob_baseline_partes
    )


    pf_total = np.concatenate(
        prob_final_partes
    )


    if coletar_amostra:

        amostra = pd.concat(
            amostra_partes,
            ignore_index=True
        )

    else:

        amostra = None


    return (
        y_total,
        pb_total,
        pf_total,
        amostra,
    )


# ============================================================
# 21. VALIDAÇÃO 2024
# ============================================================

print(
    "\nVALIDAÇÃO 2024"
)


(
    y_2024,
    prob_baseline_2024,
    prob_final_2024,
    amostra_shap_2024,
) = prever_ano(

    ano=2024,

    arquivos=ARQUIVOS[
        2024
    ],

    anos_historico=[
        2020,
        2021,
        2022,
        2023,
    ],

    coletar_amostra=True
)


np.save(
    os.path.join(
        PASTA_PREDICOES,
        "y_2024.npy"
    ),
    y_2024
)


np.save(
    os.path.join(
        PASTA_PREDICOES,
        "prob_baseline_2024.npy"
    ),
    prob_baseline_2024
)


np.save(
    os.path.join(
        PASTA_PREDICOES,
        "prob_final_2024.npy"
    ),
    prob_final_2024
)


# ============================================================
# 22. LIMIARES 2024
# ============================================================

threshold_baseline = (
    melhor_threshold_f1(
        y_2024,
        prob_baseline_2024
    )
)


threshold_final = (
    melhor_threshold_f1(
        y_2024,
        prob_final_2024
    )
)


print(
    "\nThreshold baseline:",
    threshold_baseline
)


print(
    "Threshold final:",
    threshold_final
)


# ============================================================
# 23. MÉTRICAS 2024
# ============================================================

metricas_2024 = pd.DataFrame(
    [
        calcular_metricas(
            y_2024,
            prob_baseline_2024,
            threshold_baseline,
            "RF_baseline",
            2024
        ),

        calcular_metricas(
            y_2024,
            prob_final_2024,
            threshold_final,
            "RF_final_municipal",
            2024
        ),
    ]
)


metricas_2024.to_csv(
    os.path.join(
        PASTA_TABELAS,
        "02_metricas_validacao_2024.csv"
    ),
    index=False,
    encoding="utf-8-sig"
)


display(
    metricas_2024
)


# ============================================================
# 24. MESMA SENSIBILIDADE DO BASELINE
# ============================================================

recall_baseline = float(
    metricas_2024.loc[
        metricas_2024[
            "Modelo"
        ]
        ==
        "RF_baseline",
        "Recall"
    ]
    .iloc[
        0
    ]
)


threshold_final_mesmo_recall = (
    threshold_para_recall(
        y_2024,
        prob_final_2024,
        recall_baseline
    )
)


m_base = calcular_metricas(
    y_2024,
    prob_baseline_2024,
    threshold_baseline,
    "RF_baseline",
    2024
)


m_final_mesmo = calcular_metricas(
    y_2024,
    prob_final_2024,
    threshold_final_mesmo_recall,
    "RF_final_municipal",
    2024
)


comparacao_mesmo_recall = pd.DataFrame(
    [
        m_base,
        m_final_mesmo,
    ]
)


fp_baseline = int(
    m_base[
        "FP"
    ]
)


comparacao_mesmo_recall[
    "Reducao_FP_vs_baseline"
] = (
    fp_baseline
    -
    comparacao_mesmo_recall[
        "FP"
    ]
)


comparacao_mesmo_recall[
    "Reducao_FP_pct_vs_baseline"
] = (
    comparacao_mesmo_recall[
        "Reducao_FP_vs_baseline"
    ]
    /
    fp_baseline
    *
    100
)


comparacao_mesmo_recall.to_csv(
    os.path.join(
        PASTA_TABELAS,
        "03_mesma_sensibilidade_baseline_2024.csv"
    ),
    index=False,
    encoding="utf-8-sig"
)


display(
    comparacao_mesmo_recall
)


# ============================================================
# 25. CONGELAR ESPECIFICAÇÃO ANTES DE 2025
# ============================================================

manifesto_congelado = {

    "Modelo_final":
        "RandomForestClassifier",

    "Preditores_numericos":
        NUMERICAS_FINAL,

    "Preditores_categoricos":
        CATEGORICAS_FINAL,

    "Alpha_municipio":
        ALPHA_MUNICIPIO,

    "Treino":
        "2020-2023",

    "Validacao":
        2024,

    "Threshold_final":
        threshold_final,

    "Criterio_threshold":
        "maior F1 em 2024",

    "Random_state":
        SEED,

    "Observacao":
        "Especificação congelada antes da leitura de 2025 nesta execução.",
}


with open(
    os.path.join(
        PASTA_RESULTADOS,
        "04_manifesto_modelo_congelado.json"
    ),
    "w",
    encoding="utf-8"
) as arquivo:

    json.dump(
        manifesto_congelado,
        arquivo,
        ensure_ascii=False,
        indent=2
    )


print(
    "\nMODELO CONGELADO."
)


# ============================================================
# 26. SOMENTE AGORA: PREPARAR HISTÓRICO 2024
# ============================================================

print(
    "\nCalculando estatísticas municipais de 2024 "
    "para uso histórico em 2025..."
)


ESTATISTICAS_ANUAIS[
    2024
] = calcular_estatisticas_municipais_ano(
    2024
)


# ============================================================
# 27. SOMENTE AGORA: LOCALIZAR 2025
# ============================================================

print(
    "\nAbrindo 2025 somente após congelamento."
)


ARQUIVOS[
    2025
] = localizar_arquivos_ano(
    2025
)


# ============================================================
# 28. AVALIAÇÃO TEMPORAL POSTERIOR — 2025
# ============================================================

(
    y_2025,
    prob_baseline_2025,
    prob_final_2025,
    _,
) = prever_ano(

    ano=2025,

    arquivos=ARQUIVOS[
        2025
    ],

    anos_historico=[
        2020,
        2021,
        2022,
        2023,
        2024,
    ],

    coletar_amostra=False
)


np.save(
    os.path.join(
        PASTA_PREDICOES,
        "y_2025.npy"
    ),
    y_2025
)


np.save(
    os.path.join(
        PASTA_PREDICOES,
        "prob_final_2025.npy"
    ),
    prob_final_2025
)


# ============================================================
# 29. MÉTRICAS 2025 COM LIMIARES DE 2024
# ============================================================

metricas_2025 = pd.DataFrame(
    [
        calcular_metricas(
            y_2025,
            prob_baseline_2025,
            threshold_baseline,
            "RF_baseline",
            2025
        ),

        calcular_metricas(
            y_2025,
            prob_final_2025,
            threshold_final,
            "RF_final_municipal",
            2025
        ),
    ]
)


metricas_2025.to_csv(
    os.path.join(
        PASTA_TABELAS,
        "04_metricas_avaliacao_2025.csv"
    ),
    index=False,
    encoding="utf-8-sig"
)


display(
    metricas_2025
)


# ============================================================
# 30. COMPARAÇÃO TEMPORAL FINAL
# ============================================================

comparacao_temporal = pd.concat(
    [
        metricas_2024,
        metricas_2025,
    ],
    ignore_index=True
)


comparacao_temporal.to_csv(
    os.path.join(
        PASTA_TABELAS,
        "05_comparacao_temporal_2024_2025.csv"
    ),
    index=False,
    encoding="utf-8-sig"
)


# ============================================================
# 31. CALIBRAÇÃO 2025 — MODELO FINAL
# ============================================================

brier = brier_score_loss(
    y_2025,
    prob_final_2025
)


logloss = log_loss(
    y_2025,
    prob_final_2025
)


brier_constante = brier_score_loss(
    y_2025,
    np.repeat(
        y_2025.mean(),
        len(
            y_2025
        )
    )
)


logloss_constante = log_loss(
    y_2025,
    np.repeat(
        y_2025.mean(),
        len(
            y_2025
        )
    )
)


calibracao_resumo = pd.DataFrame(
    [
        {
            "Metrica":
                "Brier",

            "Modelo_final":
                brier,

            "Preditor_constante":
                brier_constante,
        },

        {
            "Metrica":
                "Log_loss",

            "Modelo_final":
                logloss,

            "Preditor_constante":
                logloss_constante,
        },
    ]
)


calibracao_resumo.to_csv(
    os.path.join(
        PASTA_TABELAS,
        "06_calibracao_resumo_2025.csv"
    ),
    index=False,
    encoding="utf-8-sig"
)


# ------------------------------------------------------------
# Decis de calibração
# ------------------------------------------------------------

df_cal = pd.DataFrame(
    {
        "y":
            y_2025,

        "prob":
            prob_final_2025,
    }
)


df_cal[
    "Decil"
] = pd.qcut(
    df_cal[
        "prob"
    ],
    q=10,
    duplicates="drop"
)


calibracao_decil = (
    df_cal
    .groupby(
        "Decil",
        observed=True
    )
    .agg(
        N=("y", "size"),
        Escore_medio=("prob", "mean"),
        Frequencia_observada=("y", "mean"),
    )
    .reset_index()
)


calibracao_decil.to_csv(
    os.path.join(
        PASTA_TABELAS,
        "07_calibracao_decil_2025.csv"
    ),
    index=False,
    encoding="utf-8-sig"
)


# ============================================================
# 32. FIGURA — CALIBRAÇÃO
# ============================================================

fig, ax = plt.subplots(
    figsize=(
        7,
        6
    )
)


ax.plot(
    calibracao_decil[
        "Escore_medio"
    ],
    calibracao_decil[
        "Frequencia_observada"
    ],
    marker="o",
    label="RF final"
)


limite = max(
    calibracao_decil[
        "Escore_medio"
    ].max(),
    calibracao_decil[
        "Frequencia_observada"
    ].max()
)


ax.plot(
    [
        0,
        limite
    ],
    [
        0,
        limite
    ],
    linestyle="--",
    label="Calibração perfeita"
)


ax.set_xlabel(
    "Escore médio previsto"
)


ax.set_ylabel(
    "Frequência observada"
)


ax.set_title(
    "Calibração do Random Forest final — 2025"
)


ax.legend()


ax.grid(
    alpha=0.25
)


fig.tight_layout()


fig.savefig(
    os.path.join(
        PASTA_FIGURAS,
        "01_calibracao_RF_final_2025.png"
    ),
    dpi=200,
    bbox_inches="tight"
)


plt.show()


# ============================================================
# 33. CURVAS PR — 2024 E 2025
# ============================================================

for ano, y, pb, pf in [

    (
        2024,
        y_2024,
        prob_baseline_2024,
        prob_final_2024,
    ),

    (
        2025,
        y_2025,
        prob_baseline_2025,
        prob_final_2025,
    ),
]:

    precision_b, recall_b, _ = (
        precision_recall_curve(
            y,
            pb
        )
    )


    precision_f, recall_f, _ = (
        precision_recall_curve(
            y,
            pf
        )
    )


    fig, ax = plt.subplots(
        figsize=(
            7,
            6
        )
    )


    ax.plot(
        recall_b,
        precision_b,
        label=(
            "RF original "
            f"(AP={average_precision_score(y, pb):.3f})"
        )
    )


    ax.plot(
        recall_f,
        precision_f,
        label=(
            "RF final "
            f"(AP={average_precision_score(y, pf):.3f})"
        )
    )


    ax.axhline(
        y.mean(),
        linestyle="--",
        label=(
            f"Prevalência={y.mean():.3f}"
        )
    )


    ax.set_xlabel(
        "Sensibilidade"
    )


    ax.set_ylabel(
        "Precisão"
    )


    ax.set_title(
        f"Curva Precisão-Sensibilidade — {ano}"
    )


    ax.legend()


    ax.grid(
        alpha=0.25
    )


    fig.tight_layout()


    fig.savefig(
        os.path.join(
            PASTA_FIGURAS,
            f"02_curva_PR_{ano}.png"
        ),
        dpi=200,
        bbox_inches="tight"
    )


    plt.show()


# ============================================================
# 34. CURVAS ROC
# ============================================================

for ano, y, pb, pf in [

    (
        2024,
        y_2024,
        prob_baseline_2024,
        prob_final_2024,
    ),

    (
        2025,
        y_2025,
        prob_baseline_2025,
        prob_final_2025,
    ),
]:

    fpr_b, tpr_b, _ = roc_curve(
        y,
        pb
    )


    fpr_f, tpr_f, _ = roc_curve(
        y,
        pf
    )


    fig, ax = plt.subplots(
        figsize=(
            7,
            6
        )
    )


    ax.plot(
        fpr_b,
        tpr_b,
        label=(
            "RF original "
            f"(AUC={roc_auc_score(y, pb):.3f})"
        )
    )


    ax.plot(
        fpr_f,
        tpr_f,
        label=(
            "RF final "
            f"(AUC={roc_auc_score(y, pf):.3f})"
        )
    )


    ax.plot(
        [
            0,
            1
        ],
        [
            0,
            1
        ],
        linestyle="--"
    )


    ax.set_xlabel(
        "Taxa de falsos positivos"
    )


    ax.set_ylabel(
        "Sensibilidade"
    )


    ax.set_title(
        f"Curva ROC — {ano}"
    )


    ax.legend()


    ax.grid(
        alpha=0.25
    )


    fig.tight_layout()


    fig.savefig(
        os.path.join(
            PASTA_FIGURAS,
            f"03_curva_ROC_{ano}.png"
        ),
        dpi=200,
        bbox_inches="tight"
    )


    plt.show()


# ============================================================
# 35. RANKING MUNICIPAL 2020–2023
# ============================================================

(
    ranking_2024,
    _,
    _,
) = combinar_historico(
    [
        2020,
        2021,
        2022,
        2023,
    ]
)


ranking_2024[
    "Taxa_bruta_pct"
] = (
    ranking_2024[
        "Taxa_bruta"
    ]
    *
    100
)


ranking_2024[
    "Risco_suavizado_pct"
] = (
    ranking_2024[
        COL_RISCO
    ]
    *
    100
)


# ============================================================
# 36. OBTER NOMES DOS MUNICÍPIOS PELO IBGE
# ============================================================

def obter_nomes_municipios_ibge():

    url = (
        "https://servicodados.ibge.gov.br/"
        "api/v1/localidades/municipios"
    )


    try:

        resposta = requests.get(
            url,
            timeout=60
        )


        resposta.raise_for_status()


        dados = resposta.json()


        linhas = []


        for item in dados:

            codigo_7 = str(
                item[
                    "id"
                ]
            )


            codigo_6 = codigo_7[
                :6
            ]


            uf = (
                item[
                    "microrregiao"
                ][
                    "mesorregiao"
                ][
                    "UF"
                ][
                    "sigla"
                ]
            )


            linhas.append(
                {
                    COL_MUN:
                        codigo_6,

                    "Nome_municipio":
                        item[
                            "nome"
                        ],

                    "UF_IBGE":
                        uf,
                }
            )


        return pd.DataFrame(
            linhas
        )


    except Exception as erro:

        print(
            "Não foi possível consultar nomes no IBGE:"
        )

        print(
            erro
        )


        return pd.DataFrame(
            columns=[
                COL_MUN,
                "Nome_municipio",
                "UF_IBGE",
            ]
        )


nomes_ibge = (
    obter_nomes_municipios_ibge()
)


if len(
    nomes_ibge
):

    ranking_2024 = (
        ranking_2024
        .merge(
            nomes_ibge,
            on=COL_MUN,
            how="left"
        )
    )


else:

    ranking_2024[
        "Nome_municipio"
    ] = pd.NA


# ============================================================
# 37. FILTRAR N >= 500
# ============================================================

ranking_elegivel = (
    ranking_2024.loc[
        ranking_2024[
            "N"
        ]
        >=
        MIN_N_RANKING
    ]
    .copy()
)


ranking_elegivel = (
    ranking_elegivel
    .sort_values(
        COL_RISCO,
        ascending=False
    )
    .reset_index(
        drop=True
    )
)


ranking_elegivel[
    "Posicao_maior_risco"
] = (
    np.arange(
        len(
            ranking_elegivel
        )
    )
    +
    1
)


top_maiores = (
    ranking_elegivel
    .head(
        TOP_N_MUNICIPIOS
    )
    .copy()
)


top_menores = (
    ranking_elegivel
    .sort_values(
        COL_RISCO,
        ascending=True
    )
    .head(
        TOP_N_MUNICIPIOS
    )
    .reset_index(
        drop=True
    )
)


top_menores[
    "Posicao_menor_risco"
] = (
    np.arange(
        1,
        len(
            top_menores
        )
        +
        1
    )
)


ranking_elegivel.to_csv(
    os.path.join(
        PASTA_TABELAS,
        "08_ranking_municipal_completo_2020_2023.csv"
    ),
    index=False,
    encoding="utf-8-sig"
)


top_maiores.to_csv(
    os.path.join(
        PASTA_TABELAS,
        "09_top30_maiores_escores_municipais.csv"
    ),
    index=False,
    encoding="utf-8-sig"
)


top_menores.to_csv(
    os.path.join(
        PASTA_TABELAS,
        "10_top30_menores_escores_municipais.csv"
    ),
    index=False,
    encoding="utf-8-sig"
)


print(
    "\nTOP 30 — MAIORES ESCORES MUNICIPAIS"
)


display(
    top_maiores[
        [
            "Posicao_maior_risco",
            "Nome_municipio",
            "UF",
            "N",
            "S",
            "Taxa_bruta_pct",
            "Risco_suavizado_pct",
        ]
    ]
)


print(
    "\nTOP 30 — MENORES ESCORES MUNICIPAIS"
)


display(
    top_menores[
        [
            "Posicao_menor_risco",
            "Nome_municipio",
            "UF",
            "N",
            "S",
            "Taxa_bruta_pct",
            "Risco_suavizado_pct",
        ]
    ]
)


# ============================================================
# 38. DISTRIBUIÇÃO DOS EXTREMOS POR UF
# ============================================================

uf_extremos = pd.concat(
    [
        (
            top_maiores[
                "UF"
            ]
            .value_counts()
            .rename(
                "N_top30_maiores"
            )
        ),

        (
            top_menores[
                "UF"
            ]
            .value_counts()
            .rename(
                "N_top30_menores"
            )
        ),
    ],
    axis=1
).fillna(
    0
).astype(
    int
).reset_index()


uf_extremos.to_csv(
    os.path.join(
        PASTA_TABELAS,
        "11_distribuicao_top30_por_UF.csv"
    ),
    index=False,
    encoding="utf-8-sig"
)


# ============================================================
# 39. SHAP — MODELO FINAL
# ============================================================

print(
    "\nIniciando SHAP..."
)


try:

    import shap


    X_shap = (
        prep_final
        .transform(
            amostra_shap_2024
        )
    )


    nomes_features = (
        prep_final
        .get_feature_names_out()
    )


    explainer = shap.TreeExplainer(
        rf_final
    )


    shap_values = explainer.shap_values(
        X_shap
    )


    # --------------------------------------------------------
    # Compatibilidade entre versões do SHAP
    # --------------------------------------------------------

    if isinstance(
        shap_values,
        list
    ):

        valores = np.asarray(
            shap_values[
                1
            ]
        )


    else:

        valores = np.asarray(
            shap_values
        )


        if (
            valores.ndim
            ==
            3
        ):

            # n_amostras x n_features x 2 classes
            valores = valores[
                :,
                :,
                1
            ]


    # --------------------------------------------------------
    # Importância transformada
    # --------------------------------------------------------

    importancia_transformada = (
        np.mean(
            np.abs(
                valores
            ),
            axis=0
        )
    )


    shap_transformado = pd.DataFrame(
        {
            "Feature_transformada":
                nomes_features,

            "Mean_abs_SHAP":
                importancia_transformada,
        }
    )


    # ========================================================
    # MAPEAR FEATURE TRANSFORMADA PARA ATRIBUTO ORIGINAL
    # ========================================================

    def identificar_variavel_original(
        nome
    ):

        if nome.startswith(
            "num__"
        ):

            return nome.replace(
                "num__",
                "",
                1
            )


        nome_limpo = nome.replace(
            "cat__",
            "",
            1
        )


        # Ordenar do nome mais longo para evitar conflitos
        for coluna in sorted(
            CATEGORICAS_FINAL,
            key=len,
            reverse=True
        ):

            prefixo = (
                coluna
                +
                "_"
            )


            if nome_limpo.startswith(
                prefixo
            ):

                return coluna


        return nome_limpo


    shap_transformado[
        "Variavel_original"
    ] = (
        shap_transformado[
            "Feature_transformada"
        ]
        .map(
            identificar_variavel_original
        )
    )


    shap_global = (
        shap_transformado
        .groupby(
            "Variavel_original",
            as_index=False
        )[
            "Mean_abs_SHAP"
        ]
        .sum()
        .sort_values(
            "Mean_abs_SHAP",
            ascending=False
        )
        .reset_index(
            drop=True
        )
    )


    shap_global[
        "Participacao_pct"
    ] = (
        shap_global[
            "Mean_abs_SHAP"
        ]
        /
        shap_global[
            "Mean_abs_SHAP"
        ].sum()
        *
        100
    )


    shap_global.to_csv(
        os.path.join(
            PASTA_TABELAS,
            "12_SHAP_importancia_global_final.csv"
        ),
        index=False,
        encoding="utf-8-sig"
    )


    shap_transformado.to_csv(
        os.path.join(
            PASTA_TABELAS,
            "13_SHAP_features_transformadas.csv"
        ),
        index=False,
        encoding="utf-8-sig"
    )


    display(
        shap_global
    )


    # ========================================================
    # FIGURA SHAP GLOBAL
    # ========================================================

    grafico_shap = (
        shap_global
        .sort_values(
            "Participacao_pct",
            ascending=True
        )
    )


    fig, ax = plt.subplots(
        figsize=(
            9,
            6
        )
    )


    ax.barh(
        grafico_shap[
            "Variavel_original"
        ],
        grafico_shap[
            "Participacao_pct"
        ]
    )


    ax.set_xlabel(
        "Participação na importância global SHAP (%)"
    )


    ax.set_ylabel(
        "Variável"
    )


    ax.set_title(
        "Importância global SHAP — Random Forest final"
    )


    ax.grid(
        axis="x",
        alpha=0.25
    )


    fig.tight_layout()


    fig.savefig(
        os.path.join(
            PASTA_FIGURAS,
            "04_SHAP_importancia_global_final.png"
        ),
        dpi=200,
        bbox_inches="tight"
    )


    plt.show()


    # ========================================================
    # 40. SHAP POR CATEGORIA
    # ========================================================

    resumo_categorias = []


    for variavel in CATEGORICAS_FINAL:

        indices = [
            i
            for i, nome
            in enumerate(
                nomes_features
            )
            if (
                identificar_variavel_original(
                    nome
                )
                ==
                variavel
            )
        ]


        if not indices:

            continue


        contribuicao_linha = (
            valores[
                :,
                indices
            ]
            .sum(
                axis=1
            )
        )


        categorias = (
            amostra_shap_2024[
                variavel
            ]
            .astype(
                "string"
            )
            .reset_index(
                drop=True
            )
        )


        temp = pd.DataFrame(
            {
                "Categoria":
                    categorias,

                "SHAP":
                    contribuicao_linha,
            }
        )


        resumo = (
            temp
            .groupby(
                "Categoria",
                as_index=False
            )
            .agg(
                n=("SHAP", "size"),
                SHAP_medio=("SHAP", "mean"),
                SHAP_dp=("SHAP", "std"),
            )
        )


        resumo[
            "Variavel"
        ] = variavel


        resumo[
            "Erro_padrao"
        ] = (
            resumo[
                "SHAP_dp"
            ]
            /
            np.sqrt(
                resumo[
                    "n"
                ]
            )
        )


        resumo[
            "IC95_inf"
        ] = (
            resumo[
                "SHAP_medio"
            ]
            -
            1.96
            *
            resumo[
                "Erro_padrao"
            ]
        )


        resumo[
            "IC95_sup"
        ] = (
            resumo[
                "SHAP_medio"
            ]
            +
            1.96
            *
            resumo[
                "Erro_padrao"
            ]
        )


        resumo_categorias.append(
            resumo
        )


    shap_categorias = pd.concat(
        resumo_categorias,
        ignore_index=True
    )


    shap_categorias.to_csv(
        os.path.join(
            PASTA_TABELAS,
            "14_SHAP_categorias_final.csv"
        ),
        index=False,
        encoding="utf-8-sig"
    )


    # ========================================================
    # 41. SHAP NUMÉRICAS — QUANTIS
    # ========================================================

    resumo_numericas = []


    for variavel in NUMERICAS_FINAL:

        feature_nome = (
            "num__"
            +
            variavel
        )


        if (
            feature_nome
            not in nomes_features
        ):

            continue


        indice = list(
            nomes_features
        ).index(
            feature_nome
        )


        valor_shap = (
            valores[
                :,
                indice
            ]
        )


        valor_raw = pd.to_numeric(
            amostra_shap_2024[
                variavel
            ],
            errors="coerce"
        )


        temp = pd.DataFrame(
            {
                "Valor":
                    valor_raw,

                "SHAP":
                    valor_shap,
            }
        ).dropna()


        if len(
            temp
        ) < 10:

            continue


        try:

            temp[
                "Faixa"
            ] = pd.qcut(
                temp[
                    "Valor"
                ],
                q=10,
                duplicates="drop"
            )


            resumo = (
                temp
                .groupby(
                    "Faixa",
                    observed=True,
                    as_index=False
                )
                .agg(
                    n=("SHAP", "size"),
                    Valor_medio=("Valor", "mean"),
                    SHAP_medio=("SHAP", "mean"),
                )
            )


            resumo[
                "Variavel"
            ] = variavel


            resumo_numericas.append(
                resumo
            )


        except Exception:

            pass


    if resumo_numericas:

        shap_numericas = pd.concat(
            resumo_numericas,
            ignore_index=True
        )


        shap_numericas.to_csv(
            os.path.join(
                PASTA_TABELAS,
                "15_SHAP_numericas_final.csv"
            ),
            index=False,
            encoding="utf-8-sig"
        )


except Exception as erro:

    print(
        "SHAP não pôde ser concluído:"
    )

    print(
        repr(
            erro
        )
    )


# ============================================================
# 42. DISTRIBUIÇÃO ANUAL COMPLETA
# ============================================================

distribuicao_anual = []


for ano in [
    2020,
    2021,
    2022,
    2023,
    2024,
]:

    estat = (
        ESTATISTICAS_ANUAIS[
            ano
        ]
    )


    n = int(
        estat[
            "N"
        ].sum()
    )


    s = int(
        estat[
            "S"
        ].sum()
    )


    distribuicao_anual.append(
        {
            "Ano":
                ano,

            "Vinculos_ano":
                n,

            "Afastamentos":
                s,

            "Prevalencia":
                (
                    s
                    /
                    n
                ),
        }
    )


# 2025
n_2025 = len(
    y_2025
)

s_2025 = int(
    y_2025.sum()
)


distribuicao_anual.append(
    {
        "Ano":
            2025,

        "Vinculos_ano":
            n_2025,

        "Afastamentos":
            s_2025,

        "Prevalencia":
            (
                s_2025
                /
                n_2025
            ),
    }
)


distribuicao_anual = pd.DataFrame(
    distribuicao_anual
)


distribuicao_anual.to_csv(
    os.path.join(
        PASTA_TABELAS,
        "16_distribuicao_anual.csv"
    ),
    index=False,
    encoding="utf-8-sig"
)


# ============================================================
# 43. RESUMO NUMÉRICO AUTOMÁTICO PARA O TCC
# ============================================================

m24_base = metricas_2024.loc[
    metricas_2024[
        "Modelo"
    ]
    ==
    "RF_baseline"
].iloc[
    0
]


m24_final = metricas_2024.loc[
    metricas_2024[
        "Modelo"
    ]
    ==
    "RF_final_municipal"
].iloc[
    0
]


m25_final = metricas_2025.loc[
    metricas_2025[
        "Modelo"
    ]
    ==
    "RF_final_municipal"
].iloc[
    0
]


ganho_ap_abs = (
    m24_final[
        "Average_Precision"
    ]
    -
    m24_base[
        "Average_Precision"
    ]
)


ganho_ap_pct = (
    ganho_ap_abs
    /
    m24_base[
        "Average_Precision"
    ]
    *
    100
)


reducao_fp = (
    m_base[
        "FP"
    ]
    -
    m_final_mesmo[
        "FP"
    ]
)


reducao_fp_pct = (
    reducao_fp
    /
    m_base[
        "FP"
    ]
    *
    100
)


texto = f"""
RESULTADOS PRINCIPAIS — MODELO FINAL

1. Validação de 2024

O Random Forest original apresentou Average Precision de
{m24_base['Average_Precision']:.3f} e ROC-AUC de
{m24_base['ROC_AUC']:.3f}.

Com a inclusão do risco histórico municipal suavizado e do
logaritmo do volume histórico municipal, a Average Precision
passou para {m24_final['Average_Precision']:.3f} e a ROC-AUC
para {m24_final['ROC_AUC']:.3f}.

O aumento absoluto da AP foi de {ganho_ap_abs:.3f}, equivalente
a {ganho_ap_pct:.2f}% em relação ao baseline.

2. Comparação na mesma sensibilidade do baseline

Mantida sensibilidade próxima de {recall_baseline:.2%}, o modelo
final apresentou precisão de {m_final_mesmo['Precision']:.2%},
contra {m_base['Precision']:.2%} no Random Forest original.

Nessa comparação, o número de falsos positivos caiu de
{int(m_base['FP']):,} para {int(m_final_mesmo['FP']):,}, redução
de {int(reducao_fp):,} registros ({reducao_fp_pct:.2f}%).

3. Avaliação temporal posterior em 2025

Em 2025, o modelo final apresentou:
- AP = {m25_final['Average_Precision']:.3f}
- AP/prevalência = {m25_final['AP_dividida_prevalencia']:.3f}
- ROC-AUC = {m25_final['ROC_AUC']:.3f}
- precisão = {m25_final['Precision']:.2%}
- sensibilidade = {m25_final['Recall']:.2%}
- especificidade = {m25_final['Especificidade']:.2%}
- F1 = {m25_final['F1']:.3f}
- acurácia balanceada = {m25_final['Balanced_accuracy']:.3f}

O limiar utilizado em 2025 foi definido exclusivamente em 2024:
{threshold_final:.6f}.

4. Calibração em 2025

Brier do modelo final = {brier:.4f}
Brier do preditor constante = {brier_constante:.4f}

Log-loss do modelo final = {logloss:.4f}
Log-loss do preditor constante = {logloss_constante:.4f}

5. Contexto municipal

O ranking municipal foi construído com o histórico 2020–2023,
utilizando suavização em direção à taxa da respectiva UF e
alpha = {ALPHA_MUNICIPIO:.0f}.

Para a tabela principal foram considerados municípios com pelo
menos {MIN_N_RANKING} vínculos históricos.

IMPORTANTE:
Os escores municipais representam associação histórica com o
registro de afastamento por doença entre vínculos docentes e não
devem ser interpretados como medida causal de adoecimento da
população do município.
"""


with open(
    os.path.join(
        PASTA_RESULTADOS,
        "17_resumo_numerico_para_TCC.txt"
    ),
    "w",
    encoding="utf-8"
) as arquivo:

    arquivo.write(
        texto
    )


print(
    texto
)


# ============================================================
# 44. MANIFESTO FINAL
# ============================================================

manifesto_final = {

    "Base":
        "RAIS_BASE_MODELO_FINAL_V3",

    "Modelo":
        "Random Forest",

    "Treino":
        "2020-2023",

    "Validacao":
        2024,

    "Avaliacao_temporal_posterior":
        2025,

    "Preditores_numericos":
        NUMERICAS_FINAL,

    "Preditores_categoricos":
        CATEGORICAS_FINAL,

    "Municipio_direto_como_preditor":
        False,

    "Capital_interior":
        False,

    "Dummies_municipais":
        False,

    "Risco_historico_municipal":
        True,

    "Alpha":
        ALPHA_MUNICIPIO,

    "Risco_treino":
        "somente anos anteriores",

    "Risco_2024":
        "2020-2023",

    "Risco_2025":
        "2020-2024",

    "Threshold":
        threshold_final,

    "Threshold_definido_em":
        2024,

    "SHAP_n":
        SHAP_N,

    "Ranking_minimo_N":
        MIN_N_RANKING,

    "Random_state":
        SEED,
}


with open(
    os.path.join(
        PASTA_RESULTADOS,
        "18_manifesto_final.json"
    ),
    "w",
    encoding="utf-8"
) as arquivo:

    json.dump(
        manifesto_final,
        arquivo,
        ensure_ascii=False,
        indent=2
    )


# ============================================================
# 45. FINAL
# ============================================================

print(
    "\n"
    +
    "=" * 100
)

print(
    "PROCESSAMENTO FINAL CONCLUÍDO"
)

print(
    "=" * 100
)


print(
    "\nResultados:"
)

print(
    PASTA_RESULTADOS
)


print(
    "\nArquivos prioritários para enviar ao ChatGPT:"
)


for arquivo in [

    "02_metricas_validacao_2024.csv",

    "03_mesma_sensibilidade_baseline_2024.csv",

    "04_metricas_avaliacao_2025.csv",

    "05_comparacao_temporal_2024_2025.csv",

    "06_calibracao_resumo_2025.csv",

    "07_calibracao_decil_2025.csv",

    "09_top30_maiores_escores_municipais.csv",

    "10_top30_menores_escores_municipais.csv",

    "12_SHAP_importancia_global_final.csv",

    "14_SHAP_categorias_final.csv",

    "15_SHAP_numericas_final.csv",

    "16_distribuicao_anual.csv",

    "17_resumo_numerico_para_TCC.txt",

]:

    print(
        arquivo
    )